# 📊 S6E9: Predicting EV Purchases — Live Leaderboard & Meta-Analysis
**Kaggle Playground Series - Season 6, Episode 9 (`playground-series-s6e9`)**

---

### 🌟 About This Leaderboard
Below is the live leaderboard for **Predicting Electric Vehicle Purchases**.

#### 📐 Evaluation Metric & Scoring Details:
- **Primary Objective**: Binary classification to predict customer electric vehicle adoption interest (`Will_Buy_EV`).
- **Target Variable**: `Will_Buy_EV` (`0` = No EV Purchase, `1` = Will Buy EV).
- **Metric Formulation**: The leaderboard evaluates submissions using the **Area Under the Receiver Operating Characteristic Curve (ROC-AUC)** between predicted continuous probabilities and ground truth binary outcomes. 
- **Scoring Mechanics**: Predictions are ranked continuously in $[0.0, 1.0]$. The metric measures the integral area under the curve formed by plotting the True Positive Rate (Sensitivity) against the False Positive Rate (1 - Specificity) across every threshold, evaluating how well the model separates positive EV buyers from non-buyers.

This notebook displays the **Top 100 teams** in the interactive table below. The full, unabridged `leaderboard.csv` dataset containing all competitors is automatically exported and available for download in the **Output Data** section.

This notebook is scheduled to re-run **daily** to automatically reflect newly joined teams, score progressions, and updated submissions throughout the competition.

---
*Community benchmark and meta-analysis tool for competitors and organizers.*


In [ ]:
import os
import sys
import glob
import subprocess
import zipfile
import gzip
import base64
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from datetime import datetime

# Aesthetic Plot Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.dpi'] = 120

print('Environment initialized successfully.')


In [ ]:
COMPETITION_ID = 125219
COMPETITION_NAME = 'playground-series-s6e9'
DATA_DIR = './leaderboard_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Embedded full competition snapshot (backup)
EMBEDDED_B64 = "H4sIANpPmGoC/219S4/lSHbenr+CuBvZALtBBt8Fb25VVtcrM6umMmta1QsV4t7LvIzkK5uPzGauRjMyDC0sCLZkw4CEgWXLhmEZUM8DHs1Li+51T/ZPsAcz8s7+Cf5OMIK8SWZPd9d09xcnyeCJc77zivq/v/jHt7zMrPOEFy928pdTXiTWMW/as25TiKYRVXnE28Q621Y1/j7+yydVV7ZyxUlSbJL6XZPUtLYxHMsJIht/hNZbfpVya8VsFnxkxx/ZzHScRw575Nsry/449gLPs2KrJhTfJHljMKwN4yi2fWt19P54fWo+e3e6/nR96riu66zuibLDR579iDlKFAssx7F2fc7LfVfyG17KNYYrH8eN/MhaPc2vk7o0T5M+L/i1ec7Lsl/Nn4+Fj3xXC/WtwErkqnJY1NIaw5PPGYc+s1Zvq6IqE/OoK/PqavaIwSM/fGQHkzRmW7XE7yTc8OXT+X7kWpf8Vlx/0d4X4D7yg0eOfhwnondUQCOQawMv9qzVMS/Mk8p8JVpz8UJ+/MgdJYSWY+W8KKpMtEY4bHfAXOzNNS/NdZ0npfmcl7vZvtj2I8975DItJrBCK8EKTgtSwhvRsCd+GFmnSdnPvjt7xKJHTH93x7MiqySUExrxsJC5jrV6XWS8Nl/xHS9WcwHOwdeGAPwp2tvbilYYji03gzEGISc8TWozyc3XHU/5RqweUEFPfxTHpT0taEmSV8MCw3HUy+CTPX79eP5RPeeRPb6KQx81rdqrqt3UleEoHfbxWUTTklhe9ttql9yX4pOiTU8hpdzHG47S3cDFF17XpUjMl6JIyuR2NdcSiPPCSVZocYJfDmiDdguCPDsMoK9JeSnMs5bXFzzPZ5KwN/gz0pJsSKoJ3yi44Qwa60SubZ2LXIj76+NHzMOLTevxo1uChYYTqO8c+5bIoDw9L5usn78Jw5nRX9mO6dscgA3SXzowgRtYr69aAWvUii8WdgEmhk0yXKsaoYYz6GlEL/Cy66tyvzAA7JEdHjxBZGWi2FVAGo7S1TDAwV/nSdYkwvwkyXJRLoyTzR65By/iWzxPvriQWIe5nsFsZZmgYzi/Xc3NY5Fc8YXeQ9scexIU4/wCnBPWYEpRI+zTpyLP5w/h4tyOa/HmobXiNfYy46Rr1s2+S5pdVa4MptQWf0FF+LUw3/KCZ7Vo0pLfeznHdGJYSKjcJNeFEb8WDDY/MJir9shl1pYXXQ3vkPG+n28z7InjTSLugw2mTGwcYafPeZ5y87yrm65cbo8/GUj4nMBqCd1KsMEGdfUiiEnyorq/2KZv7QeHi4us5Zm4hf8wWKC2BDq8bdpdf38Toke2Nx06LHahJzW+gc0cZjBtW6Goq5dJJsxPYV/PeXchsqVptScLLQVdJpkDD9ZKuMGiQemjAD7sGX38jfmYQwWWguJHjj8JcshC7nlHKwwWDybShYV6/OKzp6fMtxeHn87f4YNsxG1SAmi4g7b6oQcDe5p80a5JYdcvzA/mZ+9mz4GdgYZEk5zQKipyEnzT4dvyJoVbVorrRbBtT1IORSt4K7bmG3F7y803vG77//fDf/2XC0sH5XOcSTSzdsmm2yflLk+axnDZ4BQZnrfvMvHZwrzE2KTDHUrFLiHkbdekwnAHkxu4oUeevYQ/Xe95fbPYa/r0kz+EJPAYgvMBjRccbG7g0wueijYVpYkD1aazA06W276nweSNCFfKRYY7qHAQedCkdZYKvGhq7qtdt+8WdAPf0HMORXG1YMAb7kAawgjKvaq7q4abddUvbJd/aHJIjITWgBquUmxYLmtdtmlV9udwWrVIyvkJd+EK2OGn4gO+VXjDHRTbxf+s1fOKC/O71ex8w85Ard1DXbrOUkANN1YPEtnW86SGp8PW76qFl/UOzDCe3bfSCWx4ijc4AfbjLBX7pO7MV1W5q7qZzRuY2Kg6AQx6M+AzCTe8gfgGYIXWyyotF3tqH+xGYHnWJUCOHzu+Y3hMkeYgtm6KvPkj+4+qrMoWZNI7sOYeniC77IMgMLxBaxlZudV6B73pufkMxmiHg8XLJf+BlrBJDngGHxbtscbwtOklYnkp4B725qdzVYulqo22guHrDtAb7IQ/CAgi7BGIQ5bS2ZqbXi86cPQOTO8ENbxAvZBDL7TBJ4HdEA1PYfeaXuwWTgDfxlOGz49j4j+bLlNrNnKJ4Snq4OLxfver//G7f/mD3/3qbxfKBvLgTXJ86wqvRH85NgsNT3HcEL579aoWt81XPzafJfU+yfuH9FYzfxLlWXm3FRdJ7cSGN5hizwVXXn317273X/2yNh/D8yXmOuurfLHXBxTejyPsVXW7h6/khm+rJ8KjrV5VhcjJdBa8XtIR/BlNIjwrI/SVBBv+oLtuGOCT5wjvQJ4/9e14aX3d0VQNDzKAb4A1/IFCxHYUWMfvjt8tzg9YszetBRXJu7y7ofihov9nfS6gPrDk+GVl+K62M8z69KPHdVfOjjboFRu9lR+HiCgg6HOB+O82xfk2fE85PPJzTUqkpzbz+5bcMYntwSdEkxzPUmCXaJqvlNkPiKZ99TeF+bxbBo043NrukggQXwRZadcjlIWIQZu9EAeKXy/4JhSFjbobwJEQUdzzfV4l14Y/xmj48e+7poOOPOfkQ8XCQDF3tDA+SLZv9SnvI8fwNeXFWV+94rddKurKPOFN14PjLRQXccmkbNLKqCWFWmH4saKu4PRnZNQRCC7Cvfjga7swD40CEmtOC7E1gsH2wrNRzCaylEN7T3lXVNfLoBxnINTSKL4pBnwp4UbgqLMNUS2obSK6+x/Zoc3x2CSBaZwRMB0QONYpvxT3F7pkdO3x09pQ+RIg2oXAVZSIggHsJr4K7NQN/lPLF4wIyq+tnR9RSFLIBZnCG4FS1iAKrTOyKP2pE8Mt3PvEAQJgeFUthvhqI7ElYY3AV0EwlGa1btqkFlVjnif1rWhmysJoP0bzBA8KF6AWtBJvBIr9xogsnuDr8/IySa7vSaEj6B+8lk18Y0PYy1SABCVGoLQ3jJm1znNe3l8fy2SL/q4who614X3ZdiU2RNnaAC+zRhzS8mYRQUBX9WaEtBl8wBnBoJ+Og51+hXCal/PPQckMfeQRFDvWVuC9y8QIB6V0vBB7/7jG/rmB/cWSIEG7Y73eoyh1RacWHDepgwi8GSvlQiMcVNPziDivy12d3JjHCc9De2nlDz5JIJ2YhOcSbYRMfV0c7fUlL7HL8zPnOpP3Csjl8AHHfCNUFIFRau0N73LzLE3yzdySEdeaPCm+ATwgwI3EGqFiBgzf9WVXJq5DL3X4RT16idF6QKlBCxTQCAf1tKWTORkId4GwrsKvqciqxaMQV7e1KBv2vdCLWr3GCBWh9Yiwn/MiqaFl5uuCAuTyAUPiupPC+fR6rVpT6SVGOBAFFsLMrZ4lVb2nY3SKn5aLtrl/lLTB9LUykL12rX05oo1QWd8YPy7t4J02opyTVNCXUZPh+LwRiF1TNDeGIu14dS2SdqHMwZju8KUT1kAjspVtI3r7vLvl4oLD8JfZji9jRzZGRr5PBjId8AyWJXJUUgtynqTV9SJrgyB8NJJYHFlbQn3RGZFitvibdbI+X/A2b/ISREJsq0jSIml5mbe8b4xIpcA8bCpcVykQ+pD2zsyZzLaMD0D0Ck6LwKS8RqTU1vHJ/1U7xBmfdFm3W7o+yu15k5QIJAnoCwm2XccxIsUGYngchK2IWtvqAqTOPIdaNkm5zP8GY0Tug3iCBVY3vBHtvkrwdiol5oPAfyLKBJHv3NBQgkOfAY9Y9sWAMyIdh8H2NM0NfEh933XZ0ls43rTYH4FGFOm8UTxYLnOLOLGtZpkeBg5LHlDLsFUaS4GNaDC0eDMZ5BJvas0jRK37mi99uDNJcolbl8OC3YA3YqWsFA0M566HroKeLpJrPgUPzoEsnDiJLwa4Eavsgo23/uTj9cen4h5BcSSVDca4kkQ4eJySA+IbsSKyiAqt1Yscbv0suYZ77PnCosDgufYj35/kgOgk1w4WGbFOhFEG4IS3aYdHM9fbNLm/yfjK5IzZoYxCwTnQRqz0V4Z3X/1xBmv0RpTtV/+tXrgQsNlR3RCpI04B+grg2oh1rcFBqECsTfA9scBS4Gc0C+PrRhMpBvcMJX8rgDbiweQ6pM83Is9BhmLfDhBZz04THUtta5kH/3wPbcSDnXV9KM8bfp3kxwK0Nquu5x+bVFCfAVnA2Ij9hu+dOHKNOFJpFhZa502HWI9/sO2HCkX6W9vk7tsBattGPKhwEGDf867CXpTMcRYZ/4MQAerpHkANx1acIQplHWWflL15mlxXO9raZdzMRlvvxaS6iVxR6gWQp7Q3puQ5sW7OFknPeEzQeBElnxQOi3UNDZ8fRxLhHYWEbZIvOAzRSfVOHmiPhwNA6CsC2yRJU4aY6GRB5aF2HlgOHlUXQzzi0szCISzJzECE0lybMpe1uIahXHf7rmkX6uZP0TJoNh5GorkEQ45O4yKIX3035ztRiNo8SgqxW5zImJyRO8ryscN9kiVlRXIUW4jxbqt3LWI8+JM1jM8iwUcx3VSwIjHMKu1raEwIMaGu8cB63v3F3Y/ufv3t9+5+fvezu1+Zd3+Jf/zx3U/vfoV//Afz7t/c/eLu7+9+gn/45d3PgPo1/vPPvv1Xq4c8aKh/mkOMUurF7rqqM1g0xx4UPXJYZL1XMW7fzV8dH8M7kMKsutv0n9vX9hUkqASwRxRQsq6LrhAmZQ16vgiKKT2j6IAHKxpYrV6RyQWGo2puQQCD8E9f/vj3P/nbf/qzP/ndz/77x/PD40yUkliwY3UIvCjygQyVePCZpAyfdyJgc3rj+PCCarkrCwUKiOWKE0cwA6ujpIRHMc+em8d8WfPD/upkgceI+O2IQuacin3KUNMOPc6JD/7h3MhDF3R871EZ19pI3BdYPARuISO2CGtaXZrH8GE+rPVDHlDX1jzmyCAQ+FzBIUuxCwdnd5eU+767SMTi6LPR9XmkFN4EpVyg4+jcGczzmz6pyv1ePMBYvVEG1XKv9pkoPNpRTStgEVYvsaO9+Xzmr/A9KGmi1rsxRV6XhEwT+QeERIrpw5O+SRBYPas7REaz8hROPRVRXC2HgoUroDm7hQhllukTr47g1Ds6qRcgDPWCDCC2dpXSuxFlVncDnku44TDNgqkW82kizOcNZYnMY/Brx1mcRJAefxTHaINvEpHKJblcAYmOSqODGMMy1lSqmR9FvJ5OuFE1AVqfSwIBbFVytQiimEpd+NQJUVzBA4h7qa4hReWPFN+VURXPL+qEQmyHKYrse5GF+LI/XmRivDG/BJBMw+/qSuxyrB2U14+J16/fvH3x5PXb03dndgzFWjILHR64sCyeteVXtdhWddk1hIcwpb0Mx+gtz0Dtmp6nC6JuT6ojKXY9QiEiUEYO/uJJVdfdVVvVzQMFWt0I4HpUV9iOUIjQtpnydbJYdm0ede0iJeOQhdMux/VkpXov4TtCCyHajeEwzZMRvK7OxG6XduZL3me8X7YneNNTuZQ2ayT8UqIhSeV8I0QObzvxyRlRuGpuKt14dKYuk9XvuhMXksFVhqOKcG5MHvkNFChJWoqHRFbffyC50zjkTH8x+jaI5EXWOwg3IMlRMa5Hxxw+1nxS7ebn3KP0ji6eDCIuAaXeBJwBVWrzAvLJp+uT9al5/vbF2evjF+/XC2+CM6VJj8soiqATAFYqGoTJ5ExcV2eKsNH/7Lc/+LP//e//o/kv/vmSbfhjesF1iD2V/OYiSfKapxAIawEX40HcoNixDRLeUyT4Ia+6hw6GPqHEVOwBCiQEqD6HAKfts/Xrdy8+vOdNg9hrkbTBM6ldJqKEL3bLqw7vJNEQFKgsIUUBKv9zDlOyjBQnF8PgUZnK/iAU3u8Nx1VJCQ8nlYgnnFd/30oMrUvaNTCZTRuREKDSEPQcPVheyudG/SA1wigNRhVUyjeLAqcCAnTvA0wsFTQpBw36QCb2gSQyJbWVIjOf6MMKuH1XIorAoQDVVOvVcsPxVCUjpprIKxhbhCb7tKmqnXiImWk/yiTLycRtUe3wlp6jsyVUz6M2IvMcxiGZ5z/lduuQADoj89NAtxIMOYpYxNi2E/6FKOYNEPhTJxCZSxmbJAPjhbPxlCIjOIF7z8Wm+1DhpO6TRXDORk1mTmRp9ACGHF2FY1QCK1O+w4aklNBbHvR49KLMkcUHTvhigNvgBLoiR49KmXzpjJMPWVLOn4qouFYhh7jzBAcakhS7wAtaq/dUGnzZleYnC8LlPpLFdi2JPhL12lx25QVRLi9Ub0fG4/XL1+bTExiQd0+PV3OdJvbnTWICq7qskgJnvUvyUlwgBIC0gW+A3jOr7XhZFq6sHN4vF9lsOmAOZbwnKEQochz7rvU6Tz56ybNq8xBt0hsNJ4dnyZNLAtZVk5TXVb4zHF2Ro6rG2Ue8+ej5wlGE09EgAgdPgaPAHcksVCXOJ4O4WlOrjvkeinm9mpclDkJayuk6cDfFFaVmKvkLJKksRkiB+VUl9iUlZ64gLJlbn8EW6kYYRrbQ0ivUAshzVdgPw7969/UPO8Rybf71lw99MR05DKKo/6Sl1rHBKMMBONb77rZbhBxTEY0WQmMAqrZtddXhSPrKItvUJ4h9KU2c2OqmycSCnnrhyJYYIj3pAkqNhiSlw3DGVs6blmXePOtFjxJNEhyNw+JQ0+vIOqtSXsy7guAQvPjwhzeEcokh+YoX+5R1W+O0897EL7lYUFoGKmBPQojvERoiVMbNJT+5JhK4q8wnKaVPeLPI6dpTswyzqXbFhxVbtcBwVBEO8RfU9cmizBGOzs2hTAvYWl81WdJu001VZUxGw4GjG75IZaWWXXKxTIW6zpjGcWSZlQozLUHxVhDDVBYFJ+j3f/3rf/qLv1+UrOyxhEcCHKtMsmr6CyJUxiKUTVdVYr6vuvu6PpBpZwy1HVngvGyu0h4qropwLiQoX83Nk6qBF21bsUj8+exgczyqoMkVhV4AeariYUN316/WZ88/MOeD7bof1nOiRR7N0bJcyrJmsAkwU64L1xuoAnJMldKnhahBmGWTbLkM2yfG5lAe00oGeEZoSFLa68I7rW67lOdm//WX+e7rL+uvvywWFWW8paZtTkwmUy7p811d0IePtJMMpwLuO5iyrLpe5JHJ8OkHi6i41fTlTmx5m7AMohS5cEG4n/SbpH7Lt1VVLtIbzth26Mgqai1h7Y3hhCrIC21qzOmGP8u5ClHdL54EOAdQiNA9abLkCO+5nXWtDG0U9wXcVPVuy+sdzlKoU2+gWOtNKi5Bz231R3jf0fqkzDpodSgbLyun91ZA4KDPoSPbTWthvk/SbsfNx0nbJPWixizTaFokde8hDO7lio1cAIGerkXJjmLR1tw84s0ewdMsA0Q5dn+kFQ5VKkAr6h3fF2SMVS3PtcNQCrqFIp10eZcte/T9sWJNUhg9FNAXArK6BtErhKmEsg2fetaVZwmoRv37H/7b33/554vQb/Itg7QGlFLiKRejKnc2VNJ6nHYItU/fzh0dWIATHErYSGBZw5iBKYWq15IY4K4qs2S36Bm1pwQMjBWMqsJhsUpaSBcpNUC05hvRJItPxaKRPpKMaPj6or0C2HAiW6e1Quu2/5yF4cI/TC2DDpW3FAxLHZ2/wRc+6/lFTVR6vcGHft61fMPhihce270vDbuqFvINMX9Vw3NiWOaUIyJo2uTiYsYdKVqckuQOFe7ugQ1H1fKYT0mTFwFlquedMFP5Tf5Q6kXqdngO+svK+yJN0yAIEC1EI5VwQcIoVTy3N+7UCUmbEioYjYcYTqTVl4KyM45YpDBfdQWvKY+3aHam5L83iaJQgVZkeoEhj4d8NZCus75pu+LDGwQ2H46S+8Rfngh2YOsjcsqXvBBdQ+kC8XkHt6FKe7Fs1z+qzOc9KPbpfCLClgXy6TRQBn9X5bnA8Ywi/TzSJYN/7U0KLzLbXj1AR31vEkJxJ+EV3HBUjY9FvnSqvDLfJLu6wgM1W5ovaCtzl5hnVdfwRZDlhmPSVSq1dYn1V7S8ITyUIla1Ezg3hIh9KpLSzLtlfx8bU/ryiFoDNEekHquBIyb7Y9+++OTd0dGLU/PtQ02tI32QMmCJLjpsegnLGKs0nAefXYgs+XC88KzxtE9hOLQgJXlVwviLreHEYwIjsh7X2JbjBV8IxhI1mTxYHoKBF8eq9ydkRAvJ8Zsv+S7ly+4F7/AVKA39lO/z5HW5fgEpviqfxrH1+vT10dO368UT2GNIJi24VZXVLqEjrmp6gQ+mcpRcf5AVo8UwQTA5hNCTrdfXslqEXw1HlfMiCmZX66YQLTdfgRK08wZU8uPh5MdlbjQbgFwuMyTXlH5KjvMkaZ1ALd7KFM8ykJ96JwZZxYAfMkKQFetJApz1p/WNKGEJqa19vxy1oARKdCDLthJakBrMtnVzY2glvKlKhBT9vJ9JNoX7owAygQdYg9kqbUxZ9iYVOAAxW/TfTPlRIhRsBGK5ntEYeniu88p8AdclykUAhDBkJBfESmzriuBCog2mK3s2DXucVZRyxoE+S8Wumomy5WRRfCAK4cyAbwhuMFXig95D1qvX786ev3hlvnp3sn5r4hgezVKBtktaNO0xVYOyipp6MmlOcRh33GCq3heF4Kurx6eP35nv3/3hi/XpM/OlWOTOhlmSSQeosNJ3XxAQdBLCVFLZpqLfCW9FDgp1zhvYjwd6eaaeUpIkK/G0oB3wkKYYNDUH3Fb4MdRr+kDedJIiU2kj1JAx/+CiqVFHpJ2gPoNZX93hSSXDuS1ax2GhwewxDedb0OOXHS8/fn8/lCYqGI39aY6s7W7TDl5EGExPyrk0AbC+FhkRwEWs5hyESAE1JHAgd7zxAzyCno4jNzp4T+KtgwNdzRNKcqrkUFLh1dsObLT/ApKYSmhLSUknE0TmcZIsXV1w74GghgqdJ0l1cSG2gueQ5+ouSsf6BGqVZGkzDwHIDEeHki4UEMs93Y9D2cMExOlJVZbwqfMJCVkedQ+keHgeXm4V2mC6kkddRLTLJRn1E97VPV+2uU1p6EEUH/CwObHBHJ0/phLned+Y5+vT9XIIyp04hXypFiwE9tRgamLOpnaQhIXgVdFi6mdKsQ0PoHCc1EWnLHDOP6m+6OpkbiAodI3v7yfBsFRpKpU1s32+bS8W1MMdK1NSN2yFMxjTWTTs31FV7l925at5ElbOcR7qlmdlmcgvYa3VWFxEua8nOLuIfecNQLF8cO1MA+pH32okBOgpzoBsxsenH5snlMVJdua6WPgz514qIKASdVEWA57KHBCnKsye4w2zfnD85puqq7dpteyF8Kaer0HalUJCkKpruF5kndQNz9t+0XUTj13gMty3igGH46YqdAj/ECNnG/hcO3xgIs4JD5drIJYHOiVtU+6pSsyimneMDdOXo/MJZMQHKCEhQXEE24NFqqurK5CPJn0gzToSpYDquRMUIgaFDIOBulMLNY09c5p0WT1QXx6DPUhyraHnOlN4SIs1UXZGovys6udDAkNroH/wUI6iyXsCQ11dg6nqXBRSm9ZZUojUPMffbxc+hkjLvQ1qkiJtCWowVZgLXOr9fgeDCk+12TRdsUg/UZ7z3gOR+W0HMOTo7K8fyxE3vPMH/ICqW9QhqGF33CNPTcQBPqANpkbimAf3WVbzfOfQZOhO65kE7W62WOnpAUqZThOFecaLLsnNVx+v5ofR9SZ2HbgymSaKRsIzSFKd7RQaXdIIwjwb7rKJGAeUL+153WUioxFQV6mtB8r7mXgvTsRs5pd6ocIDW0Drb0WPHSB3rStvCMes9+/erx/Q1mkpk1nrnotbapSRU6S68BbB82b8mjuLru2psi5zqc4Aw0qdTkCIszpKoAtn0NpsmYj0x1KAFABeTo1NgBpMDbhF+ArWe+xIWVG/2PwV3PHKg0FAr5EQoMfapE+keeBWmLKuni6nHA66cCmzjPM2LNgPeEhTahngv0oPPi/TEJkf58pIhq9xOGF6zo3mr6gPHLv8/v180IOyI9G03tFIOVSsHHyAj0JTpxRz9BzRVrHJkwV1oZrIgSQiUWrFpVwAeYqnUjvwd8768k19z0lpRjlpNsXXWVPAkqVJDfbsKd10yc+85D3i6FIs20Pj6fv4xKEueY+wQngQoLSTWttWnyAU3ZlvqIDYtJ25vpyXRmR1Yoy8fCphXdCaK7WEX8Llq2qaF1MW/NUwAS7dX7mQ5h74PV92XRcyjcY8nc+NZB02xx6vFhzVGQdoh7dqLm+SHCzO1/064LoUOvX9bbvoQHInTZM/WQMN5uvLG6iLSuzjKJxTFtedjqxcDNi1wKZyBDY1x3fxmbLlnisD0AL8FsZLmI+r2wX5Att3xj2NhggUC2C9xKaCTVeVM5/87zmNjF6Jqm35AzP1Y/pWimknLIR4KrymkOhMyDtHTvjtbTWvHhEp8ifF9alBppH4YoBDlhopdoY0MCi35O7ma7x7+VCdY9otOVOMFTJaqwgPcdrAUoboLe8z81WdtLfNNl0GBNG9FLx8uBorsmmBwVSJzaPi+6uqalqRUWfQLMHnT33XsqJHnc4IRRwXIZ8qtHkUrb88WTdbvhUbXi9SOmzKn0kRlziZCmswVWoLZa//uhbXnfkkTW5T0T+Q0ZBuXYvyhzT3dbfVeIMFitFSmxQYpXkGZasR7dYPDOC5EwXzKcFcUMGuKPiu4U0Hu15CQwNNFIKhu7xNkxvz/bKtNDo4J54cdNplFGdDAlPvF9uS+1xCw9NlTkSWEbUEd8gPXjZA2nSNBQLCCFHKWHkLqXcfL5zQm5ln8KDpQwVsapYKDvSA5O71soZW9bQIglV7JdkC2fkrbz4ot4i2uoX7OfSkdIKtayzYjHhIGxssI9q0fFuZ36mqB6+UOJBDSQBAP69o2wMdGGDbv/kPv/nTb/7um1/hf//zN983v/lPv/mTb7785pff/PSbX3zzD9/83W++v0hWUW5gPOPkDPYCAUfaNDSGlAuOIFTNzMWSRh6Jal+Zbz82P0nqOhE1X+bu/SmG8mUOmJZcIIZS5TnYMFiMuz+/+8XdT+9+/u0f3/393a/ufvLt98y7v/n2e3c/+/ZP7n6Nf/Hrux89yFMPHha602/qa7GB41MTdjaV0Kus6zaOMz/n7jTuO6xWOIOpGTsooGOdgbOmPJsrB10OEh8ubgZc7MDnqZG6wCejAJeLr2u+hbrsRL+8XME+yJF75GWu5IJa4mFXQz16D3r7or3KZ7kYYtneZN49ijKFhGHpmG2gVEq97xrzVVKXi1HLg54a2WdiUSdUkwEKGaoDM6Dc72ORQe9bc71LQR5rscg6sOnGEdn0Ym2GFVwtgDzFSFxyXO8ol8zNU75LkmKZNMMB9A+FdRJeSjQkqcSDRykMTlXkqmkSUZr0gwqxyNFIR6EbKmWfkHWwSi8yWKj024MOHdOVS9t0Hms5U4OnbF2yOO87UCYHHHAapwusc8RyH7meN8+e+dMVXsP6FkDgsDrWdJi6tavBqprrvOSXO36z9KTUAKE9KbFPeZEJreF6icH0kF3ApFHJqrYyP+kQfoD+L4qqbjgFx4NAueBC4Q0WqQagmO7EOOr6pDRP9/TLsn91mtOWfW+UvwCrFGUCWglBTHe5eBZ8fNbPpmBJK4OJaksJfMBhsU5YUBj6lu9Tfi2Dj4LD9dKo0nzkgo19q7KTD06d1gzdfbQCIj19FY+8WmWz6c0XpRy6bRe8kvqNnUkcOALhhYLDhOg5vBCfd/Xqq19SfXNdf/3l1z9cjrkcVO88MpAZwOTf1fRdFMIvkgNthkBrVhHwp2y1O8xtt+mNaMFUWnBNPYUX0MzR465BMPHm7YuTp7PvblPRT7eOOXLCa0Ng2KICR0KV6wI7PmDc1Emz4yVirTJpFoyDUfpKd2w5ciziusu3vKRWYUhUE9CyDe5lVXIavTefdQ11l+8XFgpqPubPXbrW4VIt2asVBlOFOpeC+e8+S7Ksmvs2mhzWOkBBs3W9JxiWqmZjmvZYnfJbuuGtuubLIU6ySOEkwbMKIbZZgsBNl+eocfB5UsiLIjZzOge/MWZ2Kei3Uo2EAKXQlNmTwXlwb0pM5iDsqfma1nsjEMs97e9iuqkGIaX5XJR80TURU8ppVDdKW1jpMKtMYIPFmnmDPpxxcaV6uN8mu13/et5czOStAaMwV7rBcU1Na6jtiemBPJpvOKJJxRQaRIEkzQbMThaMxlijd4dcgVrQDAsMpgp5IWUJ+fbGn0/DRZSsGbPNriwZSxiWqnZ5j4zXZ9TfZb5JFjNw1KI0fWhKwVm3hL0CFDLU5RLEmF/xGr5o7pB9d3oHSgdamYQVVQevbrhqHC+g+URizsKWE6Wzi5LiKahh1CszIiHA0XcA2GSp6OQcNVV3u/zYbLobQg5ywE7RPVcSDEIBUUwTTmrX+1xsQFQWcjw5E6g1j0lDDGgNJFVWXVtfhUIc4VzIltCS+iAWHTve5PekmFbIyiPNBckOd6l68vKaKgXbF221emjSUasck+mKKm0ICQm+JvkUgpprKM01r+ezY3QK3YlZsIG9aSykBOrykICussTpvqzwt7kpOBhko/qBdwCFCH2VD3nxp8MoYC6uvv6ynMebFCf6Uz7coTY96ojbER5oyFKtEiG1oxyLxny+m9+vSCw7GuctaVyMWTleCECs1xlkIqMnlOlYhDtDamxqsJVTb3AkImcILPgQ6biOChBdnwLE2oSLrNpFTors7Pg2lGFLyz2sdF+WJUQoYhwghliXe7xjySnxN+f1bMoB2UOudcQarq7JRQEVdJNrML7vijxPdhy7bB4n1bIB3benpkNsI1wsLbseVmGrsQZyXU2TGF1VSCXVRc+OO4WENunezYDD4vF6lFh1ZvaLLLIjlVfPUZMA7E6S5GlVZZCgLK9NLQXvu3JvvuKFedw9MFs+XV5nyzbTHuiMF3lHb6HLcY5MLJZbqs6L+SCPI3tMXS3Et+6DIWZs8aGPDcv7eZeYZ0lJOaiFlfGnb27LKbliWNAMeEgbdDiWfYJZnyfmOe/zqp73CDrssDvZlv2hBG8V2nD12J2srqwzGh5e395io6o/2P/Bok85llfXBVqcvHeU1uCAGS7TUR7Vdo+7Lcdm53m1LKxMkyJ2TB89BzYnKGToJAeMl6zcfFyldGvAsi3TGy0OCXEGtAJDjtZpnNaneVLys+4qqT+tilkHhxyJHtPHtmzwTAjfEP6G8BCm56JB27jLC34+62QbqIQ2X3ZE5EwCW/BgV5Xw7AiM+3WRiNd7hEmLyqY31v/tiMYwKiArQhquutMyiHH+T/l1kgzJuEW7kz/mx2zZe1USNikdT7Ztu0x3wcNAvKllv/GiU40dvAXdBVvkBMNaNbo/9OfIkCjZwbVdcr4wDs5UXxqeo1D4muCGy3RAR03xz6orYSoGXN7v4RqaffyRotFMuWPtsUDfUor38g1XlfSoSEqhC2XfuPm2u+Bi90DtPRgJlh0MxzRpHGiva+s7IGIQtSLJ2mreyALSPO6OTOw3Aw6L9a2ssv2uK4YBfCj7ctiUuhH0C/kRXf7VFbKnyqG7q9WMHTx/KBvac+xx1/Jrvix2ssOZBdsfmu1oLk7jIc3V9SSf+kF4+REMUrpo07dHI2zLKL5Iqpu8M1xdzvNlJwLtufmkQjwGV7i4+YSymNMFErYrL+dutw0onuBY0EKcmumX3/05cVBZ3xJZuvSdweiwbDlamUp4UXLKVLhjgY+aBdIq/7Cen+vhLhSdzKLrJkF8gZTVeVdV+FyKMnZdlvEHBqxGLZF8piq7urrlKY6k3BlVkyYy0LQ0w/yB7M5y7DrQW+LGsax4SLDEQooivjYY0tHRd76zaEN19IF2Y1mZr8rLmuiQvsRSaskJ/Lh5KvitIPaYLk4QFUv1MwQ0aYAQiUf4A3JUnObhmK6miwu+/WP5/35k3v3Vtz/49nvyYoOfL9PEkc5DurFPgTLfiYy57U1aN4Y7XXBJ9aD3T4/M5+vTz9bHa3N9/MI8e75+vsgBQH/UpruU6/Gspk92YNe3oEykQimHIdLTdkEYWe+rDop4ccbzpPhwUc85Ns1MuUqeTFH0A74hPMQldWK4ukBIHbSr8+5//fxvSwTi6fJ2lFEjYQNlv01HEUw6sBU9bkd+Z3WWdhsYPPOUXNKCKnuhPv9uJK8l2Pabqt6XBIYglaIY7jfg12KHGP72ViymwVis0xRuGJFm7Qi8J6zhjjdfsoFs0P2xZ5QrSpqb+f2gw6yGrXsdXJr2JJNNi5qDNYaryoQRucHVZ2lV7lNeUWSyf+gG60keebNbCfeqnrqx3fE6TOjjs64+E0U998s0ban9shvK1td9VzeJZJa+rQLWkHJgeEoRLTJdni6Ru5QyYhpnuLpaGMgCA85CDxu7LMHix6vEvRsO2RaCMm/D93ThHeQwfdukJ2e7Mplw4de1yBfXkMnBGmVkpTiH5rtoxV4tMKT7cWDImbXhDbHz5EMz/u4LH2+b8Zac6COHych+jJiUSDyT7kkO5SXLIv3oNAHxPV8MRdF1SGOSGstDOTss0hLwtqNvpC/TDOj1jgrR1r35xNsky1QS080tSs5OgrfAGq4qPDJX/p4FT7+7PjVfrU9frY8+e7ocIxzvZ4Ig6Zp7EOi62vbbvGpr+eGV4ad+4W2/Lzu6vHDDZ/2YRGTGcTpYIiYrGQdoCNJDfTC966y8f8eaNJsUoaunkV4ZRJdgWdcUFWKUPrkmHVDDIx7NaVGkZJ6AV9X94ohRWkeXJVzJCC0KkgqFNlxf3xZA+U2+2dHvZlBU3aXYrB4YdNBHX4YVFuB01TmBDVfN97mU5KRaibz73Vw38irOZXfASDtdJjuTrtSKAQ9xKjtN6cBnFc7Abv5i1Eyovr9sACJGRX3yvC0SnkOCvm6TjNFL6o42j3nx0PCR6kl0mQwpL+GsI9dwx0ojXWZKs8en1QOtX468cUC9ik1nnO4hLCXUcFVRMaAhmaO07q4lP7Mpsp+T6LHNk4WyQrwj+JWGQ5SvWro9bQXN92KfLLPQkR5JlfMvzmACe0AhQ1t56ut+KQrzWd0tuwwPPSLzZWfMpSj2EgoZumYuG0l0wvd8MctCMzoDG2d08wuzLlvyWYhuqCcXclS/pydvQ23PbhBylPNEqxvr3+PBGYJb/LhGIiFAD6iC1v32+z/97Q/++rff/8ff/uDHCz3RrfZR5MZDavOqaulVVHkwokaO1YuLNoFhrE2eUsywZJk6rQIx0vEJtcAZFkDcoLKRTzd6npBN4+ZJV7bLCrL8bUHUVUV0WYwjM9C0oFB4SBt/i5qYvlVDiYSbaplGkyn/QRAbWgskNkTgpcqGvi8vZSQeLTJyyUd049vCUIw33dFNf0Rb0nHFjhY4seHqeznp+tvVG6h4Izu0lpQqVn44CmP5Wy5Qk12y7/q8pn0fu0Nd6/XmttwvWqddzTLAND0ZlUoYluqLDCkePPvql3nS031h77/+Mi/47TLqClV1N2IUVbhWk/dF2efFLUSFeogSh+F5Aj02jwTVIRZJU7rUd3gbO6bkkm2lBN9JtOGqQmFMXoIc+zYpW9E1MoNFlRUTQe+mull+OkpoKbm2LNFdj4sdm868upIzptvIfv9Xf/r7n/6X3/3n//p/fvSTudfxmTomNHrgyTKkyATdTURGSA/5Ee9/T/FMRrcPEvlfMvZBxUNP9jJa/SHacPXIX+DTmEedXXadvPZpXoKigrw0QT5Vu+V9MBPYcPVNnXaMJ3gMav7yhfnd9du1+ebt+mx9ZB49fbZepKZsPWiiZW7Azi/FNXTwCmpI6fhx7C+WDbsU5p3I7tvL5b30OvM2SKOImsK8AX0JUbpYTiWJgQ4/zvEtv/rxchCauLVz+GQ7vv3i0jb+P1DYjxT0agAA"

lb_df = None

# 1. Real-time dynamic fetch via Kaggle Leaderboard Service API (No auth required)
try:
    print('Fetching real-time dynamic leaderboard from Kaggle Service API...')
    req = urllib.request.Request(
        'https://www.kaggle.com/api/i/competitions.LeaderboardService/GetLeaderboard',
        data=json.dumps({'competitionId': COMPETITION_ID}).encode('utf-8'),
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)', 'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=15) as resp:
        api_data = json.loads(resp.read().decode('utf-8'))
        teams_map = {t['teamId']: t for t in api_data.get('teams', [])}
        
        records = []
        for entry in api_data.get('publicLeaderboard', []):
            t_id = entry.get('teamId')
            team_info = teams_map.get(t_id, {})
            records.append({
                'Rank': entry.get('rank'),
                'TeamName': team_info.get('teamName', f'Team_{t_id}'),
                'Score': float(entry.get('displayScore', 0) or 0),
                'SubmissionCount': team_info.get('submissionCount', 1),
                'LastSubmissionDate': team_info.get('lastSubmissionDate')
            })
            
        if records:
            lb_df = pd.DataFrame(records)
            print(f'⚡ Real-time live fetch successful! Total dynamic teams: {len(lb_df):,}')
except Exception as e:
    print(f'Live API note: {e}')

# 2. Fallback to CLI download if available
if lb_df is None or len(lb_df) == 0:
    try:
        cmd = [sys.executable, '-m', 'kaggle', 'competitions', 'leaderboard', '-c', COMPETITION_NAME, '-d', '-p', DATA_DIR, '-q']
        res = subprocess.run(cmd, capture_output=True, timeout=15)
        if res.returncode == 0:
            zip_path = os.path.join(DATA_DIR, f'{COMPETITION_NAME}.zip')
            if os.path.exists(zip_path):
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(DATA_DIR)
            csv_files = glob.glob(f'{DATA_DIR}/*.csv')
            if csv_files:
                lb_df = pd.read_csv(csv_files[0])
                print(f'⚡ Kaggle CLI fetch successful! Total teams: {len(lb_df):,}')
    except Exception as e:
        pass

# 3. Fallback to embedded snapshot
if lb_df is None or len(lb_df) == 0:
    decompressed = gzip.decompress(base64.b64decode(EMBEDDED_B64)).decode('utf-8')
    lb_df = pd.read_csv(io.StringIO(decompressed))
    print(f'✅ Full Leaderboard Snapshot loaded! Total teams: {len(lb_df):,}')


In [ ]:
# Standardize column names
lb = lb_df.copy()
if 'Score' in lb.columns:
    lb['Score'] = pd.to_numeric(lb['Score'], errors='coerce')
    lb = lb.sort_values(by=['Score', 'LastSubmissionDate'], ascending=[False, True]).reset_index(drop=True)
    lb['Rank'] = range(1, len(lb) + 1)

top_score = lb['Score'].max()
median_score = lb['Score'].median()
lb['Gap_To_1st'] = top_score - lb['Score']

if 'LastSubmissionDate' in lb.columns:
    # Format date strictly as 'Fri Aug 28 18:14:17 2026'
    lb['SubmissionDateTime'] = pd.to_datetime(lb['LastSubmissionDate'], errors='coerce')
    lb['Last_Best_SubmissionDate'] = lb['SubmissionDateTime'].dt.strftime('%a %b %d %H:%M:%S %Y').fillna(lb['LastSubmissionDate'])
else:
    lb['Last_Best_SubmissionDate'] = 'N/A'

# Save full clean leaderboard to disk for participants
output_lb_path = 'leaderboard.csv'
lb.to_csv(output_lb_path, index=False)
print(f'Full leaderboard saved to {output_lb_path} ({len(lb):,} entries)')


## 🏆 Top 100 Live Leaderboard Table
Below is the live interactive ranking table displaying the Top 100 teams, ROC-AUC scores, absolute gap to 1st place, and last best submission timestamps:


In [ ]:
display_cols = ['Rank', 'TeamName', 'Score', 'Gap_To_1st', 'Last_Best_SubmissionDate']

top_100 = lb[display_cols].head(100).copy()

# Format styling with custom dark headers and gradient highlights
styled_table = top_100.style.format({
    'Score': '{:.5f}',
    'Gap_To_1st': '{:.5f}'
}).background_gradient(
    subset=['Score'], cmap='Blues'
).set_properties(**{
    'font-family': 'monospace',
    'font-size': '12px',
    'text-align': 'left'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', '#f8fafc'), ('font-weight', 'bold')]},
    {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
])

display(styled_table)


## 📈 Executive Summary & Key Milestones
Statistical summary and key percentile score boundaries across the entire competitor pool:


In [ ]:
# Summary statistics
total_teams = len(lb)
p90_score = lb['Score'].quantile(0.90)
p75_score = lb['Score'].quantile(0.75)
p50_score = lb['Score'].quantile(0.50)

summary_html = f"""
<div style="background: linear-gradient(135deg, #1e293b, #0f172a); color: #f8fafc; padding: 24px; border-radius: 12px; font-family: sans-serif; box-shadow: 0 4px 14px rgba(0,0,0,0.3); margin-top: 10px;">
    <h2 style="margin-top:0; color: #38bdf8; border-bottom: 1px solid #334155; padding-bottom: 8px;">📊 Competition Snapshot</h2>
    <div style="display: flex; justify-content: space-around; flex-wrap: wrap; margin-top: 16px;">
        <div style="background: #334155; padding: 14px 20px; border-radius: 8px; text-align: center; min-width: 140px; margin: 6px;">
            <div style="font-size: 12px; color: #94a3b8; text-transform: uppercase;">Total Teams</div>
            <div style="font-size: 24px; font-weight: bold; color: #f8fafc;">{total_teams:,}</div>
        </div>
        <div style="background: #334155; padding: 14px 20px; border-radius: 8px; text-align: center; min-width: 140px; margin: 6px;">
            <div style="font-size: 12px; color: #94a3b8; text-transform: uppercase;">Top Score (1st)</div>
            <div style="font-size: 24px; font-weight: bold; color: #4ade80;">{top_score:.5f}</div>
        </div>
        <div style="background: #334155; padding: 14px 20px; border-radius: 8px; text-align: center; min-width: 140px; margin: 6px;">
            <div style="font-size: 12px; color: #94a3b8; text-transform: uppercase;">Top 10% Cutoff</div>
            <div style="font-size: 24px; font-weight: bold; color: #38bdf8;">{p90_score:.5f}</div>
        </div>
        <div style="background: #334155; padding: 14px 20px; border-radius: 8px; text-align: center; min-width: 140px; margin: 6px;">
            <div style="font-size: 12px; color: #94a3b8; text-transform: uppercase;">Top 25% Cutoff</div>
            <div style="font-size: 24px; font-weight: bold; color: #fbbf24;">{p75_score:.5f}</div>
        </div>
        <div style="background: #334155; padding: 14px 20px; border-radius: 8px; text-align: center; min-width: 140px; margin: 6px;">
            <div style="font-size: 12px; color: #94a3b8; text-transform: uppercase;">Median Score</div>
            <div style="font-size: 24px; font-weight: bold; color: #cbd5e1;">{median_score:.5f}</div>
        </div>
    </div>
</div>
"""
display(HTML(summary_html))


## 📊 Score Distribution & Progression Curves
KDE distribution curve and rank-vs-score progression trajectory:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Score Density & Threshold Distribution
ax1 = axes[0]
sns.histplot(lb['Score'], bins=40, kde=True, color='#3b82f6', ax=ax1, edgecolor='black', alpha=0.6)
ax1.axvline(top_score, color='#22c55e', linestyle='--', linewidth=2, label=f'1st Place ({top_score:.5f})')
ax1.axvline(p90_score, color='#06b6d4', linestyle=':', linewidth=2, label=f'Top 10% ({p90_score:.5f})')
ax1.axvline(p50_score, color='#f59e0b', linestyle='-.', linewidth=2, label=f'Median ({p50_score:.5f})')
ax1.set_title('Public Leaderboard Score Distribution (ROC-AUC)', fontweight='bold')
ax1.set_xlabel('ROC-AUC Score')
ax1.set_ylabel('Number of Teams')
ax1.legend(loc='upper left', frameon=True)

# Plot 2: Rank vs Score Progression Curve
ax2 = axes[1]
ax2.plot(lb['Rank'], lb['Score'], color='#6366f1', linewidth=2.5, label='Team Score Curve')
ax2.scatter(lb['Rank'].iloc[:15], lb['Score'].iloc[:15], color='#ef4444', zorder=5, label='Top 15 Teams')
ax2.fill_between(lb['Rank'], lb['Score'], lb['Score'].min(), color='#6366f1', alpha=0.1)
ax2.set_title('Score vs Rank Progression Curve', fontweight='bold')
ax2.set_xlabel('Leaderboard Rank')
ax2.set_ylabel('ROC-AUC Score')
ax2.set_xlim(1, len(lb))
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()


## 🎯 Top 50 Gap-to-1st Place Tier Dynamics
Analysis of the fine margins separating the leading competitors from 1st place:


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
top50 = lb.head(50)

bars = ax.bar(top50['Rank'], top50['Gap_To_1st'] * 10000, color=sns.color_palette("viridis_r", len(top50)), edgecolor='none')
ax.set_title('Top 50 Teams: Absolute Gap to 1st Place (in 0.0001 Units)', fontweight='bold')
ax.set_xlabel('Leaderboard Rank')
ax.set_ylabel('Gap to 1st Place (x 10⁻⁴ ROC-AUC)')
ax.grid(True, linestyle='--', alpha=0.5)

# Highlight Top 10
ax.axvspan(0.5, 10.5, color='#22c55e', alpha=0.1, label='Top 10 Tier (< 0.00035 Gap)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


## 💡 Strategic Takeaways & Modeling Insights

1. **The Sub-Decimal Precision Frontier (`0.94600+`)**:
   - Because the synthetic dataset contains rounding boundaries across decimal place values, standard tree splits often struggle unless augmented with **Decimal Digit Decomposition** ($10^{-4}$ to $10^3$) or fine-grained **Quantile KBins Discretization** ($N \ge 100$).
2. **Variance Reduction via Multi-Seed Averaging**:
   - Ensembling multiple random seeds (e.g. 5–10 seeds) of high-resolution **LightGBM** (`max_bin: 1024`, `num_leaves: 247`, `max_depth: 5`) significantly reduces test ranking variance without dilution from weaker architectures.
3. **Zero-Leakage Nested Target Encoding**:
   - Target encoding with an inner Stratified K-Fold loop (with dual smoothing $\alpha = 10.0, 50.0$) provides orthogonal lift while guaranteeing zero fold leakage.


## 📚 Citation & Competition Reference
> Yao Yan, Walter Reade, Elizabeth Park. *Predicting Electric Vehicle Purchases*. https://kaggle.com/competitions/playground-series-s6e9, 2026. Kaggle.
